# 01 — Exploration: Order Fulfillment & Delivery Performance

Exploratory analysis of the Olist Brazilian E-Commerce dataset, loaded from the star schema
built by `etl/transform_load.py`. This notebook answers: what does the data actually look like,
what's the shape/scale, and what data-quality issues are visible before deeper analysis.

In [1]:
import sys, os
sys.path.insert(0, os.path.join("..", "etl"))
sys.path.insert(0, os.path.join("..", "sql"))
sys.path.insert(0, os.path.join("..", "analysis"))
import pandas as pd
from db import get_conn

conn = get_conn()
fact_orders = pd.read_sql("SELECT * FROM fact_orders", conn)
print(fact_orders.shape)
fact_orders.head()

(99441, 24)


,order_id,customer_id,seller_id,order_status,date_key,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,...,payment_value,payment_installments,payment_type,review_score,processing_days,carrier_pickup_days,shipping_days,delivery_delay_days,on_time_flag,timestamp_anomaly_flag
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,3504c0cb71d7fa48d967e0e4c94d59d9,delivered,20171002,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,...,38.71,1.0,voucher,4.0,0.007431,2.366493,6.062650,-7.107488,1.0,0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,289cdb325fb7e7f891c38608bf9e0962,delivered,20180724,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,...,141.46,1.0,boleto,4.0,1.279745,0.462882,12.039410,-5.355729,1.0,0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,4869f7a5dfa277a7dca6462dcf3b52b2,delivered,20180808,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,...,179.12,3.0,credit_card,5.0,0.011505,0.204595,9.178113,-17.245498,1.0,0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,66922902710d126a0e7d26b0e3805106,delivered,20171118,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,...,72.20,1.0,credit_card,5.0,0.012419,3.745833,9.450498,-12.980069,1.0,0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,2c9e548be18521d1c43cde1c582c6de8,delivered,20180213,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,...,28.62,1.0,credit_card,5.0,0.042940,0.893113,1.937824,-9.238171,1.0,0


## Overall delivery performance

In [2]:
delivered = fact_orders[fact_orders.order_status == "delivered"]
on_time_pct = 100 * delivered.on_time_flag.mean()
print(f"Delivered orders: {len(delivered):,}")
print(f"Overall on-time rate: {on_time_pct:.2f}%")
print(f"Average delay (days, negative=early): {delivered.delivery_delay_days.mean():.2f}")

Delivered orders: 96,478
Overall on-time rate: 91.89%
Average delay (days, negative=early): -11.18


## Distribution of delivery delay (IQR outlier method)

In [3]:
from delay_analysis import iqr_outliers
stats, outliers = iqr_outliers(fact_orders)
stats

{'q1': np.float64(-16.243819444444444),
 'q3': np.float64(-6.388252314814815),
 'iqr': np.float64(9.85556712962963),
 'lower_bound': np.float64(-31.02717013888889),
 'upper_bound': np.float64(8.39509837962963),
 'n_outliers': 4869,
 'pct_outliers': 5.05}

In [4]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8,4))
delivered.delivery_delay_days.clip(-40, 40).hist(bins=60, ax=ax)
ax.set_title("Delivery delay distribution (clipped at +/-40 days for readability)")
ax.set_xlabel("delivery_delay_days (negative = early)")
plt.savefig("delay_distribution.png", dpi=110, bbox_inches="tight")
plt.show()
print("saved delay_distribution.png")

saved delay_distribution.png


## Seller-level anomaly detection (z-score)

In [5]:
from delay_analysis import seller_zscore_anomalies
anomalies = seller_zscore_anomalies(fact_orders)
print(f"Sellers evaluated: {len(anomalies)}  |  Flagged anomalous: {anomalies.is_anomalous.sum()}")
anomalies.head(10)

Sellers evaluated: 1217  |  Flagged anomalous: 135


,seller_id,order_count,avg_delay,avg_review,z_score,is_anomalous
2356,cb41bfbcbda0aea354a834ab222f9a59,11,10.299938,3.272727,5.831021,True
2037,b1b3948701c5c72445495bd161b83a4c,14,4.737372,1.928571,4.335047,True
473,2a1348e9addc1af5aaa619b1a3679d6b,45,2.627415,3.272727,3.767603,True
442,26e2c91ef821e1ff8985f408788fe35b,12,2.237865,3.000000,3.662839,True
56,054694fa03fe82cec4b7551487331d74,20,-0.309476,3.400000,2.977767,True
1127,5f67c6082caacb26e431a7b17940cece,13,-0.807937,3.538462,2.843713,True
906,4e5725ba188db8252977a4f0227bd462,21,-1.228281,3.523810,2.730667,True
2770,f08c008c8a8d31417763738a1788a2a8,17,-1.709464,3.750000,2.601260,True
30,02d35243ea2e497335cd0f076b45675d,12,-2.374151,2.666667,2.422502,True
2340,ca4b77513ac2040591b0d8fae6958380,14,-2.537173,4.500000,2.378659,True


## Correlation check: does payment structure relate to delay or satisfaction?

In [6]:
from delay_analysis import payment_delay_correlation
payment_delay_correlation(fact_orders)

{'installments_vs_delay_corr': np.float64(-0.032),
 'freight_vs_delay_corr': np.float64(-0.051),
 'delay_vs_review_corr': np.float64(-0.267)}

## Takeaway

See `reports/findings_and_recommendations.md` for the full write-up. The short version: on-time
rate is ~92% overall but volatile month to month, delay is concentrated in a small number of
statistically anomalous sellers, delay correlates negatively with review score (not with payment
structure), and delivery risk is geographically concentrated in Brazil's North/Northeast states.